# End-to-End Pipeline Testing — Code Generation v2

Full pipeline: Steps 1–5 (preprocessing) → Step 6 (code generation) → execution

**Required uploads:**
- `sf_film_May7_2025_data.gpkg` — the film locations dataset
- `code_generation_v2.md` — the code generation prompt

**Rate limiting:** Gemini free tier = 10 RPM. LLM calls use an 8s delay.


In [27]:
!pip install geopandas google-genai shapely


In [28]:
import json
import re
import time
import copy
import unicodedata
import difflib
import textwrap
from typing import Dict, Any, List, Optional, Tuple
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd

from google import genai
from google.genai import types


In [29]:
# === CONFIGURATION ===

# Model: Gemini 2.5 Flash      | RPM:5   TPM: 250K  RPD: 20  |  gemini-2.5-flash
# Model: Gemini 2.5 Flash Lite | RPM: 10 TPM: 250K  RPD: 20  |  gemini-2.5-flash-lite
# Model: Gemini 3 Flash        RPM: 5      TPM: 250K     RPD: 20  | gemini-3-flash-preview
# Model: Gemini 3.1 Flash Lite RPM: 0 / 15   TPM: 250K  RPD: 500 |  gemini-3.1-flash-lite-preview



from google.colab import userdata
MODEL_NAME = "gemini-2.5-flash"
MODEL_NAME = "gemini-3-flash-preview"
MODEL_NAME = "gemini-2.5-flash-lite"
MODEL_NAME = "gemini-3.1-flash-lite-preview"
LLM_DELAY = 8  # seconds between LLM calls (rate limit)

# Load API key from Colab Secrets
# To set this up: click the 🔑 icon in the left sidebar → add a secret named GEMINI_API_KEY

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in Colab Secrets. Add it via the 🔑 sidebar."
print("API key loaded successfully.")


client = genai.Client(api_key=GEMINI_API_KEY)


API key loaded successfully.


In [30]:
# === LOAD DATA ===
from google.colab import drive
drive.mount('/content/drive')
DB_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/database/SQLite-stuff/sf_film_May7_2025_data.gpkg'

gdf = gpd.read_file(DB_PATH)
print(f"Loaded {len(gdf)} rows, {len(gdf.columns)} columns")
print(f"Columns: {list(gdf.columns)}")
print(f"Unique films: {gdf.drop_duplicates(subset=['Title','Year']).shape[0]}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 2084 rows, 11 columns
Columns: ['id', 'Title', 'Year', 'Locations', 'Fun_Facts', 'Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3', 'geometry']
Unique films: 337


## Street/Place/Avenue/Boulevard/etc normalizer

In [31]:
"""
Street suffix normalizer for SF film locations.

One-way transformation from canonical / full forms to short, period-less
abbreviations. Apply this to BOTH sides for matching to work cleanly:

    - the user's query        (call from Step 1 normalizer)
    - the Locations column    (call once at data load time)

Rules covered (6):
    Street / Streets / St. / St      ->  St
    Avenue / Avenues / Ave. / Ave    ->  Ave
    Boulevard / Boulevards / Blvd.   ->  Blvd
    Place / Places / Pl. / Pl        ->  Pl
    Lane  / Lanes   / Ln.  / Ln      ->  Ln
    Road  / Roads   / Rd.  / Rd      ->  Rd

Saint guard:
    'St' and 'St.' are ambiguous — they can mean Street (trailing) or
    Saint (leading). To avoid mangling "St. Germain" into "St Germain"-
    as-a-Street, the abbreviated Street pattern requires a letter
    immediately before the 'S', separated by a single space. This
    correctly rejects:
        "100 St. Germain"    (preceded by digit)
        "St. Francis"        (preceded by start-of-string)
    and correctly accepts:
        "Market St."         (preceded by 't' from Market)
        "1 Front St"         (preceded by 't' from Front)
        "Vallejo st"         (preceded by 'o' from Vallejo)

Non-goals:
    - Drive is NOT normalized. The only 'Dr.' in the data is 'Dr. Carlton
      B. Goodlett Pl.' (Doctor, not Drive). All actual Drive occurrences
      in the dataset are spelled out, so no rule is needed.
"""

import re


# Order matters. Full forms run first (they're unambiguous); the
# abbreviated Street form with its Saint guard runs after. For the
# other suffixes, order is cosmetic but kept consistent.
#
# Each rule is (compiled_pattern, replacement_string).

NORMALIZATION_RULES = [
    # --- Street ---
    # Full forms (Street, Streets) — no ambiguity with Saint.
    (re.compile(r'\bStreets?\b', re.IGNORECASE), 'St'),
    # Abbreviated forms (St, St.) — Saint guard via lookbehind requiring
    # [letter][space] immediately before. (?!\w) ensures we don't match
    # into a longer word like "Stockton".
    (re.compile(r'(?<=[a-zA-Z]\s)St\.?(?!\w)', re.IGNORECASE), 'St'),

    # --- Avenue ---
    (re.compile(r'\bAvenues?\b', re.IGNORECASE), 'Ave'),
    (re.compile(r'\bAve\.?(?!\w)', re.IGNORECASE), 'Ave'),

    # --- Boulevard ---
    (re.compile(r'\bBoulevards?\b', re.IGNORECASE), 'Blvd'),
    (re.compile(r'\bBlvd\.?(?!\w)', re.IGNORECASE), 'Blvd'),

    # --- Place ---
    (re.compile(r'\bPlaces?\b', re.IGNORECASE), 'Pl'),
    (re.compile(r'\bPl\.?(?!\w)', re.IGNORECASE), 'Pl'),

    # --- Lane ---
    (re.compile(r'\bLanes?\b', re.IGNORECASE), 'Ln'),
    (re.compile(r'\bLn\.?(?!\w)', re.IGNORECASE), 'Ln'),

    # --- Road ---
    (re.compile(r'\bRoads?\b', re.IGNORECASE), 'Rd'),
    (re.compile(r'\bRd\.?(?!\w)', re.IGNORECASE), 'Rd'),
]


def normalize_street_suffixes(text):
    """
    Apply all suffix normalization rules to a single string.

    Returns the input with recognized street suffixes converted to
    their short, period-less abbreviations. Leaves 'St.' untouched
    when it is almost certainly 'Saint' (preceded by a digit or
    start-of-string rather than a street name).

    Safe on None or empty input (returns input unchanged).
    """
    if not text:
        return text
    for pattern, replacement in NORMALIZATION_RULES:
        text = pattern.sub(replacement, text)
    return text




In [32]:
# must run to make this happen: Street/Place/Avenue/Boulevard/etc normalizer

# gdf = gpd.read_file('sf_film_May7_2025_data.gpkg')
gdf['Locations'] = gdf['Locations'].apply(normalize_street_suffixes)

## manual testing

In [ ]:
# gdf[
#     gdf['Director'].astype(str).str.lower().str.contains('hitchcock', na=False)
# ].drop_duplicates(subset=['Title', 'Year'])


result = gdf[gdf['Locations'].astype(str).str.lower().str.contains('market st', na=False)].drop_duplicates(subset=['Title', 'Year'])

len(result)

29

### T2


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

def process_sf_film_query(gdf):
    """
    Execute a read-only query against the SF film locations GeoDataFrame.
    Returns a standardized result dictionary.
    """
    def clean_column_data(series):
        """Remove empty/null/whitespace/stringy-null values from a string Series.

        WARNING: Returns a FILTERED series with fewer rows.
        Use ONLY for final output cleanup, NEVER for building boolean masks.
        """
        stringified = series.astype(str)
        exclude = (
            series.isna() |
            (stringified.str.strip() == '') |
            (stringified.str.lower().isin(['none', 'nan', 'null']))
        )
        return series[~exclude]

    try:
        gdf_copy = gdf.copy()

        task_results = {}

        # ---- Task t1: retrieve films shot on market street ----
        # Predicate: Locations contains 'market street'
        mask_t1_locations = gdf_copy['Locations'].astype(str).str.contains(
            'market st', case=False, na=False
        )
        t1_matched_rows = gdf_copy.loc[mask_t1_locations]

        # Derive text_view based on response_granularity: 'film'
        t1_text_view = t1_matched_rows.drop_duplicates(subset=["Title", "Year"])

        # Prepare records for the result, cleaning up relevant columns for display
        t1_records = t1_text_view[["Title", "Year", "Director", "Writer", "Actor_1", "Actor_2", "Actor_3"]].copy()
        for col in ["Title", "Director", "Writer", "Actor_1", "Actor_2", "Actor_3"]:
            if col in t1_records.columns:
                t1_records[col] = clean_column_data(t1_records[col])
        t1_records_list = t1_records.to_dict("records")

        task_results["t1"] = {
            "matched_rows": t1_matched_rows,
            "text_view": t1_text_view,
            "result": {
                "records": t1_records_list,
                "count": len(t1_text_view),
            }
        }

        # ---- Assemble final result ----
        final_data = task_results["t1"]["text_view"]
        final_summary = f"Found {len(final_data)} films shot on Market Street."
        if len(final_data) > 0:
            sample_titles = final_data['Title'].head(3).tolist()
            if len(final_data) > 3:
                final_summary += f" For example: {', '.join(sample_titles[:-1])}, and {sample_titles[-1]}.\n"
            else:
                final_summary += f" For example: {', '.join(sample_titles)}."

        result = {
            'data': final_data.to_dict(orient='records'),
            'summary': final_summary,
            'metadata': {
                'query_type': 'retrieve',
                'task_count': 1,
                'task_ids': ['t1'],
                'offer_map': True
            }
        }

        # ---- Log (UTF-8 safe) ----
        with open('code_gen_result.log', 'a', encoding='utf-8', errors='replace') as f:
            f.write('=' * 50 + '\n')
            f.write('Query Result\n')
            f.write('-' * 50 + '\n')
            if isinstance(result['data'], list):
                for record in result['data']:
                    f.write(str(record) + '\n')
            else:
                f.write(str(result['data']) + '\n')
            f.write(f"Summary: {result['summary']}\n")
            f.write('=' * 50 + '\n\n')

        return result

    except Exception as e:
        error_result = {
            'data': None,
            'summary': f"Error processing query: {str(e)}",
            'metadata': {
                'error': str(e),
                'error_type': type(e).__name__
            }
        }
        with open('code_gen_result.log', 'a', encoding='utf-8', errors='replace') as f:
            f.write('=' * 50 + '\n')
            f.write(f'ERROR: {str(e)}\n')
            f.write('=' * 50 + '\n\n')
        return error_result



process_sf_film_query(gdf)

{'data': [{'id': 49,
   'Title': 'The Net',
   'Year': 1995,
   'Locations': 'Montgomery & Market Streets',
   'Fun_Facts': '',
   'Director': 'Irwin Winkler',
   'Writer': 'John Brancato',
   'Actor_1': 'Sandra Bullock',
   'Actor_2': 'Jeremy Northam',
   'Actor_3': '',
   'geometry': <POINT (-122.402 37.789)>},
  {'id': 55,
   'Title': 'Terminator - Genisys',
   'Year': 2015,
   'Locations': '1 Market St. Landmark Building',
   'Fun_Facts': '',
   'Director': 'Alan Taylor',
   'Writer': 'James Cameron',
   'Actor_1': 'Arnold Schwarzenegger',
   'Actor_2': 'Jason Clarke',
   'Actor_3': '',
   'geometry': <POINT (-122.395 37.794)>},
  {'id': 68,
   'Title': 'The Game',
   'Year': 1997,
   'Locations': 'Sheraton Palace Hotel (639 Market Street)',
   'Fun_Facts': 'The hotel was destroyed in the 1906 earthquake and fire, had to be rebuilt, and was reopened in 1909.',
   'Director': 'David Fincher',
   'Writer': 'John Brancato',
   'Actor_1': 'Michael Douglas',
   'Actor_2': 'Sean Penn',
 

## known values mechanics

In [54]:
# === KNOWN VALUES (for fuzzy matching) ===
def extract_known_values(gdf):
    known = {}
    for col in ['Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3']:
        vals = gdf[col].dropna().astype(str).str.strip()
        vals = vals[(vals != '') & (~vals.str.lower().isin(['none','nan','null']))]
        known[col] = sorted(vals.unique().tolist())
    known['Actor'] = sorted(set(known.pop('Actor_1') + known.pop('Actor_2') + known.pop('Actor_3')))
    for col in ['Title', 'Locations']:
        vals = gdf[col].dropna().astype(str).str.strip()
        vals = vals[(vals != '') & (~vals.str.lower().isin(['none','nan','null']))]
        known[col] = sorted(vals.unique().tolist())
    return known

known_values = extract_known_values(gdf)
print(f"Known values: { {k: len(v) for k, v in known_values.items()} }")


Known values: {'Director': 276, 'Writer': 294, 'Actor': 627, 'Title': 336, 'Locations': 1628}


In [55]:
# === LLM HELPER ===
# def call_gemini(system_prompt, user_input, temperature=0):
#     response = client.models.generate_content(
#         model=MODEL_NAME,
#         contents=user_input,
#         config=types.GenerateContentConfig(
#             system_instruction=system_prompt,
#             response_mime_type="application/json",
#             temperature=temperature,
#         ),
#     )
#     return response.text


# === LLM HELPER ===
import os
import time

from google.protobuf.json_format import MessageToDict

RAW_RESPONSE_DIR = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/raw_responses'
os.makedirs(RAW_RESPONSE_DIR, exist_ok=True)

def call_gemini(system_prompt, user_input, temperature=0, label=None):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=user_input,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            temperature=temperature,
        ),
    )

    # Capture raw response for post-mortem debugging
    _save_raw_response(response, label)

    return response.text




def _save_raw_response(response, label):
    """Save the raw Gemini response. If dict conversion fails, saves as string."""
    try:
        os.makedirs(RAW_RESPONSE_DIR, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S')
        safe_label = (label or 'unknown').replace('/', '_').replace(' ', '_')
        path = os.path.join(RAW_RESPONSE_DIR, f'{ts}_{safe_label}.json')

        response_data = None

        # Attempt 1: The most common internal dict conversion
        try:
            if hasattr(response, 'to_dict'):
                response_data = response.to_dict()
            elif hasattr(response, '_result'):
                response_data = type(response._result).to_dict(response._result)
        except:
            pass

        # Attempt 2: If it's a proto-message but sneaky
        if not response_data:
            try:
                from google.protobuf.json_format import MessageToDict
                response_data = MessageToDict(response)
            except:
                pass

        with open(path, 'w', encoding='utf-8') as f:
            if response_data:
                # If we successfully got a dict, save as pretty JSON
                json.dump(response_data, f, indent=4)
            else:
                # THE FALLBACK: Save the string representation
                # This ensures you NEVER lose the data even if the object is weird
                f.write(str(response))

        print(f"Successfully saved to {path}")

    except Exception as e:
        print(f'   [Total failure: {repr(e)}]')



---
## Step 1: Query Normalizer (rule-based)


In [56]:
CITY_PATTERNS = [
    re.compile(r'\bin\s+san\s+francisco\b', re.I),
    re.compile(r'\bin\s+sf\b', re.I),
    re.compile(r'\bin\s+the\s+city\b', re.I),
    re.compile(r'\bin\s+california\b', re.I),
    re.compile(r'\bsan\s+francisco\b', re.I),
    re.compile(r'\bsf\b', re.I),
]

SYNONYMS = {
    "movie": "film", "movies": "films",
    "acted in": "appeared in", "stars": "actors",
    "shot at": "filmed at", "filmed in": "filmed at",
    "screenwriter": "writer", "screenwriters": "writers",
}

STOP_WORDS = {
    'a','an','the','in','on','at','to','for','of','by','with','and','or','but',
    'is','was','are','were','been','be','have','has','had','do','does','did',
    'will','would','could','should','may','might','can','shall','that','this',
    'these','those','it','its','my','me','i','we','us','our','you','your',
    'he','she','him','her','his','they','them','their','what','which','who',
    'whom','where','when','how','why','if','then','than','so','not','no','nor',
    'from','about','each','every','all','both','few','many','much','more','most',
    'some','any','such','into','over','after','before','between','also','just',
    'only','very','too','here','there',
}

DOMAIN_WORDS = {
    'film','films','movie','movies','show','shows','directed','written','acted',
    'starring','featured','filmed','shot','actor','actors','actress','director',
    'directors','writer','writers','location','locations','scene','scenes','set',
    'list','find','search','count','how','many','compare','top','most','least',
    'best','worst','first','last','near','within','around','mile','miles','km',
    'year','years','decade','during','between','from','appeared','made','produced',
    'released',
}

FUZZY_CUTOFFS = {'Director':0.75,'Writer':0.75,'Actor':0.75,'Title':0.80,'Locations':0.65}

def is_numeric(word):
    return bool(re.match(r'^\d+s?$', word))

def extract_content_clusters(words):
    clusters, start = [], None
    for i, w in enumerate(words):
        wl = w.lower()
        if wl in STOP_WORDS or wl in DOMAIN_WORDS or is_numeric(wl):
            if start is not None:
                clusters.append((start, i)); start = None
        else:
            if start is None: start = i
    if start is not None: clusters.append((start, len(words)))
    return clusters

def fuzzy_match_cluster(cluster_text, known_values):
    candidates = [cluster_text]
    if cluster_text.lower().startswith('the '): candidates.append(cluster_text[4:])
    else: candidates.append('the ' + cluster_text)
    for col in ['Locations','Actor','Director','Writer','Title']:
        if col not in known_values: continue
        cutoff = FUZZY_CUTOFFS.get(col, 0.75)
        for cand in candidates:
            matches = difflib.get_close_matches(cand, known_values[col], n=1, cutoff=cutoff)
            if matches:
                ratio = difflib.SequenceMatcher(None, cand.lower(), matches[0].lower()).ratio()
                if ratio >= cutoff: return (matches[0], col, cluster_text)
            if col in ('Director','Writer','Actor') and ' ' not in cand:
                for full_name in known_values[col]:
                    parts = full_name.split()
                    if len(parts) >= 2:
                        ratio = difflib.SequenceMatcher(None, cand.lower(), parts[-1].lower()).ratio()
                        if ratio >= cutoff: return (full_name, col, cluster_text)
    return None

def normalize_query(query, known_values):
    corrections = []
    q = unicodedata.normalize('NFKC', query).strip().lower()
    q = re.sub(r'\s+', ' ', q)
    for p in CITY_PATTERNS: q = p.sub('', q)
    q = re.sub(r'\s+', ' ', q).strip()
    for old, new in SYNONYMS.items(): q = q.replace(old, new)
    words = q.split()
    clusters = extract_content_clusters(words)
    offset = 0
    for start, end in clusters:
        adj_s, adj_e = start + offset, end + offset
        cluster_text = ' '.join(words[adj_s:adj_e])
        match = fuzzy_match_cluster(cluster_text, known_values)
        if match:
            matched_value, col, original = match
            if matched_value.lower() != cluster_text.lower():
                corrections.append({'original': cluster_text, 'corrected': matched_value, 'column': col})
                new_words = matched_value.lower().split()
                words[adj_s:adj_e] = new_words
                offset += len(new_words) - (end - start)

    # NEW: final canonical pass on suffixes
    normalized = normalize_street_suffixes(' '.join(words)).lower()
    return {'normalized': normalized, 'corrections': corrections}


---
## Step 2: Safety Gate (rule-based)


In [57]:
SAFE_PHRASES = [
    "add locations to","add location to","remove duplicates","drop duplicates",
    "drop me a list","update me","alter my search","change my search",
    "modify my search","insert a filter","delete filter","remove filter",
]
BLOCKED_PHRASES = [
    "save changes","add a new","add new","change the",
    "export modified","write to database","write to db",
]
BLOCKED_VERBS = [re.compile(rf'\b{v}\b') for v in
    ['delete','update','drop','insert','remove','modify','alter','edit','truncate','overwrite']]

def _find_spans(text, patterns):
    spans = []
    for p in patterns:
        if isinstance(p, re.Pattern):
            for m in p.finditer(text): spans.append((m.start(), m.end()))
        else:
            s = 0
            while True:
                idx = text.find(p, s)
                if idx == -1: break
                spans.append((idx, idx + len(p))); s = idx + 1
    return spans

def _is_covered(span, safe_spans):
    return any(span[0] >= ss and span[1] <= se for ss, se in safe_spans)

def safety_check(query):
    q = query.lower()
    safe_spans = _find_spans(q, SAFE_PHRASES)
    blocked = []
    for phrase in BLOCKED_PHRASES:
        for s, e in _find_spans(q, [phrase]): blocked.append((s, e, phrase))
    for verb_re in BLOCKED_VERBS:
        for m in verb_re.finditer(q): blocked.append((m.start(), m.end(), m.group()))
    for start, end, blocked_by in blocked:
        if not _is_covered((start, end), safe_spans):
            return {'safe': False, 'blocked_by': blocked_by,
                    'message': "This operation cannot be performed as it would modify the database. Only read-only operations are permitted."}
    return {'safe': True, 'blocked_by': None, 'message': None}

---
## Step 3: Task Decomposer (hybrid: short-circuit + LLM)


In [58]:
MULTI_INTENT_PATTERNS = [
    re.compile(r'\bcompare\b', re.I), re.compile(r'\bhow many\b', re.I),
    re.compile(r'\bwhy\b', re.I), re.compile(r'\band which\b', re.I),
    re.compile(r'\band how\b', re.I), re.compile(r'\band what\b', re.I),
    re.compile(r'\band where\b', re.I), re.compile(r'\band list\b', re.I),
    re.compile(r'\band show\b', re.I), re.compile(r'\band their\b', re.I),
    re.compile(r'\bthen\b', re.I), re.compile(r'\balso\b', re.I),
    re.compile(r'\bas well as\b', re.I), re.compile(r'\bplus\b', re.I),
]

TASK_DECOMPOSER_PROMPT = """You are a query decomposer for a San Francisco film locations database.

Your ONLY job: break a user query into the fewest atomic semantic tasks needed to answer it.

## Output Schema (JSON)
{"tasks": [{"id": "t1", "kind": "retrieve | compare | rank | count | explain | clarify", "source": "exact contiguous text span from the query", "dependsOn": []}]}

## Rules
1. Decomposition ONLY. Do NOT extract filters, mention columns, or reference implementation.
2. Use contiguous spans from the user's query for "source". Do not rephrase.
3. Multi-condition ≠ multi-task. "Films by Hitchcock from the 1950s at North Beach" is ONE retrieve.
4. Max 4 tasks. IDs sequential: t1, t2, t3, t4.
5. dependsOn must reference earlier task IDs only. Omit or use [] if none.
6. Valid kinds: retrieve, compare, rank, count, explain, clarify.
7. No schema words (table, column, sql, join, filter, dataframe, merge) in source.

## Examples
Query: "films directed by hitchcock"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films directed by hitchcock","dependsOn":[]}]}

Query: "what films shot in tenderloin and how many are there"
{"tasks":[{"id":"t1","kind":"retrieve","source":"what films shot in tenderloin","dependsOn":[]},{"id":"t2","kind":"count","source":"how many are there","dependsOn":["t1"]}]}

Query: "compare films in chinatown with ones around north beach"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films in chinatown","dependsOn":[]},{"id":"t2","kind":"retrieve","source":"films around north beach","dependsOn":[]},{"id":"t3","kind":"compare","source":"compare films in chinatown with ones around north beach","dependsOn":["t1","t2"]}]}

Query: "actors who appeared in the most films in 1982"
{"tasks":[{"id":"t1","kind":"rank","source":"actors who appeared in the most films in 1982","dependsOn":[]}]}

Query: "top 5 directors with the most films"
{"tasks":[{"id":"t1","kind":"rank","source":"top 5 directors with the most films","dependsOn":[]}]}

Query: "films within 1 mile radius of coit tower"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films within 1 mile radius of coit tower","dependsOn":[]}]}

Query: "how many films were shot in the 80s"
{"tasks":[{"id":"t1","kind":"count","source":"how many films were shot in the 80s","dependsOn":[]}]}

Query: "films by hitchcock from the 1950s at north beach"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films by hitchcock from the 1950s at north beach","dependsOn":[]}]}

Query: "what is this database about"
{"tasks":[{"id":"t1","kind":"clarify","source":"what is this database about","dependsOn":[]}]}
"""

VALID_KINDS = {'retrieve','compare','rank','count','explain','clarify'}
SCHEMA_LEAK = {'table','column','sql','join','filter','dataframe','merge','index','row','select'}

def validate_task_plan(plan, query):
    if 'tasks' not in plan or not isinstance(plan['tasks'], list): return False
    tasks = plan['tasks']
    if len(tasks) == 0 or len(tasks) > 4: return False
    seen = set()
    for i, t in enumerate(tasks):
        if not all(k in t for k in ('id','kind','source')): return False
        if t['id'] != f"t{i+1}": return False
        seen.add(t['id'])
        if t['kind'] not in VALID_KINDS: return False
        if not t.get('source','').strip(): return False
        if set(t['source'].lower().split()) & SCHEMA_LEAK: return False
        for dep in t.get('dependsOn', []):
            if dep not in seen or dep == t['id']: return False
    return True

def make_fallback(query):
    return {"tasks": [{"id":"t1","kind":"retrieve","source":query,"dependsOn":[]}]}

def decompose_query(query):
    if not any(p.search(query) for p in MULTI_INTENT_PATTERNS):
        return make_fallback(query)

    for p in MULTI_INTENT_PATTERNS:
      if p.search(query):
          print(f"  [decomposer] matched pattern: {p.pattern}")
          break

    try:
        # resp = call_gemini(TASK_DECOMPOSER_PROMPT, query)
        resp = call_gemini(TASK_DECOMPOSER_PROMPT, query, label=f'step3_{query}')

        plan = json.loads(resp)
        if validate_task_plan(plan, query): return plan
        print("  [decomposer] Validation failed, fallback")
        return make_fallback(query)
    except Exception as e:
        print(f"  [decomposer] Error: {e}, fallback")
        return make_fallback(query)


---
## Step 4: Filter Extractor (LLM) + Post-processors


In [59]:
FILTER_EXTRACTOR_PROMPT = """You are a filter extractor for a San Francisco film locations database.

## Input Guarantees
The query has already been normalized (lowercase, city terms removed, typos corrected).
You receive a task plan with tasks, each having id, kind, source, and dependsOn.

## Schema
Columns: Title (str), Year (int64), Locations (str), Fun_Facts (str), Director (str), Writer (str), Actor_1 (str), Actor_2 (str), Actor_3 (str), geometry (Point)
- Actor is a VIRTUAL field meaning "search Actor_1, Actor_2, Actor_3 with OR logic"

## Capabilities
The system can: filter rows, count, rank/sort, group by column, compute frequencies.

## MOST IMPORTANT RULE: Constraint vs Output Dimension
A predicate defines what must be TRUE of a row. It does NOT encode what the query asks about.
- "Films by Hitchcock" → Director IS a constraint (predicate: Director == hitchcock)
- "Which directors filmed on Geary St" → Director is the OUTPUT DIMENSION, NOT a constraint. Only Locations is a constraint.
- "Top 5 directors with the most filming locations in 1982" → Only Year is a constraint. Director is the ranking output.

## Predicate Format
Each task gets a `predicate` field: either null or a recursive boolean tree.
Logic node: {"logic": "AND"|"OR", "clauses": [...]}
Leaf clause: {"field": "<col>", "op": "<op>", "value": "<val>", "type": "attribute"|"spatial"}

## Allowed Ops
==        exact match (person, title, single year)
contains  partial text match (Locations, Title, Fun_Facts)
between   inclusive range [low, high] (Year)
> < >= <= numeric comparison (Year)
within_distance  spatial proximity (geometry field, value: {reference_place, distance, unit})

## Allowed Fields
Title, Year, Locations, Fun_Facts, Director, Writer, Actor (virtual), geometry

## Field Mapping Rules
1. Person names (director/writer/actor) → use == with lowercase value
2. Location/street/neighborhood names → Locations contains
3. Film titles → Title contains or ==
4. Year/decade/era → Year with ==, between, or comparison ops
5. "near X" / "around X" / "at X" → Locations contains (NOT spatial)
6. "within N miles of X" → geometry within_distance (IS spatial)
7. Fun facts / trivia → Fun_Facts contains
8. Actor/starring → field: "Actor" (virtual, expanded later)

## Dependency Rules
1. If a task has dependsOn, read the parent's source text for context but do NOT copy parent predicates.
2. Dependent count/rank tasks with no additional constraints get predicate: null.
3. If a dependent task adds its own constraint (e.g., "why is vertigo..."), extract only that new constraint.

## Task Kind Guidance
- retrieve: extract all row constraints from source
- count: extract constraints if present, null if purely referential
- rank: extract only filtering constraints (year, location), NOT the ranking dimension
- compare: usually null (operates on dependency results)
- explain: extract any specific entity reference (e.g., a film title)
- clarify: always null

## Examples

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films directed by alfred hitchcock"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films directed by alfred hitchcock","predicate":{"field":"Director","op":"==","value":"alfred hitchcock","type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films by alfred hitchcock from the 1950s on sutter st"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films by alfred hitchcock from the 1950s on sutter st","predicate":{"logic":"AND","clauses":[{"field":"Director","op":"==","value":"alfred hitchcock","type":"attribute"},{"field":"Year","op":"between","value":[1950,1959],"type":"attribute"},{"field":"Locations","op":"contains","value":"sutter st","type":"attribute"}]}}]}

Input: {"tasks":[{"id":"t1","kind":"rank","source":"top 5 directors with the most filming locations in 1982"}]}
Output: {"tasks":[{"id":"t1","kind":"rank","source":"top 5 directors with the most filming locations in 1982","predicate":{"field":"Year","op":"==","value":1982,"type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"which directors filmed on geary st"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"which directors filmed on geary st","predicate":{"field":"Locations","op":"contains","value":"geary st","type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films with clint eastwood on geary st"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films with clint eastwood on geary st","predicate":{"logic":"AND","clauses":[{"field":"Actor","op":"==","value":"clint eastwood","type":"attribute"},{"field":"Locations","op":"contains","value":"geary st","type":"attribute"}]}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films within 1 mile of coit tower"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films within 1 mile of coit tower","predicate":{"field":"geometry","op":"within_distance","value":{"reference_place":"coit tower","distance":1,"unit":"mile"},"type":"spatial"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films near golden gate bridge"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films near golden gate bridge","predicate":{"field":"Locations","op":"contains","value":"golden gate bridge","type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films shot on valencia st","dependsOn":[]},{"id":"t2","kind":"count","source":"how many are there","dependsOn":["t1"]}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films shot on valencia st","predicate":{"field":"Locations","op":"contains","value":"valencia st","type":"attribute"},"dependsOn":[]},{"id":"t2","kind":"count","source":"how many are there","predicate":null,"dependsOn":["t1"]}]}

Input: {"tasks":[{"id":"t1","kind":"rank","source":"actors who appeared in the most films in the 90s"}]}
Output: {"tasks":[{"id":"t1","kind":"rank","source":"actors who appeared in the most films in the 90s","predicate":{"field":"Year","op":"between","value":[1990,1999],"type":"attribute"}}]}

## Strict Rules
1. Only extract ROW CONSTRAINTS. Never encode ranking, grouping, or output shape.
2. Preserve id, kind, source, and dependsOn exactly as received.
3. Values stay lowercase as received from normalization.
4. Do not invent fuller name forms. Use the value as-is.
5. One predicate per task. Use logic nodes for multiple constraints.
6. Null predicate is valid and expected for dependent/compare/clarify tasks.
7. Actor is always virtual — never emit Actor_1/Actor_2/Actor_3 directly.
8. Only use within_distance for explicit distance language.
9. Do not add constraints not stated in the source text.
10. between always uses a 2-element array [low, high].
11. Return the complete tasks array with all original fields preserved.
12. Output valid JSON only.
"""

VALID_FIELDS = {'Title','Year','Locations','Fun_Facts','Director','Writer','Actor','geometry'}
VALID_OPS = {'==','contains','between','>','<','>=','<=','within_distance'}

def _validate_predicate(pred):
    if pred is None: return True
    if 'logic' in pred:
        if pred['logic'] not in ('AND','OR'): return False
        if 'clauses' not in pred: return False
        return all(_validate_predicate(c) for c in pred['clauses'])
    # Leaf clause
    if not all(k in pred for k in ('field','op','type')): return False
    if pred['field'] not in VALID_FIELDS: return False
    if pred['op'] not in VALID_OPS: return False
    if pred['op'] == 'between' and (not isinstance(pred.get('value'), list) or len(pred['value']) != 2): return False
    if pred['op'] == 'within_distance':
        if pred['field'] != 'geometry' or pred['type'] != 'spatial': return False
        v = pred.get('value', {})
        if not all(k in v for k in ('reference_place','distance','unit')): return False
    return True

def validate_filter_result(result, input_plan):
    if 'tasks' not in result: return False
    input_tasks = input_plan['tasks']
    output_tasks = result['tasks']
    if len(output_tasks) != len(input_tasks): return False
    for inp, out in zip(input_tasks, output_tasks):
        if out.get('id') != inp['id']: return False
        if out.get('kind') != inp['kind']: return False
        if out.get('source') != inp['source']: return False
        if not _validate_predicate(out.get('predicate')): return False
    return True

# def extract_filters(task_plan, user_query):
#     input_json = json.dumps(task_plan)
#     try:
#         # resp = call_gemini(FILTER_EXTRACTOR_PROMPT, input_json)
#         resp = call_gemini(FILTER_EXTRACTOR_PROMPT, input_json, label=f'step4_{user_query[:30]}')

#         result = json.loads(resp)
#         if validate_filter_result(result, task_plan):
#             return result
#         print("  [filter_extractor] Validation failed, null predicates fallback")
#         print(f"  [filter_extractor] Rejected payload: {json.dumps(result, indent=2)[:500]}")

#     except Exception as e:
#         print(f"  [filter_extractor] Error: {e}, null predicates fallback")
#     # Fallback: null predicates
#     fallback = copy.deepcopy(task_plan)
#     for t in fallback['tasks']: t['predicate'] = None
#     return fallback

def extract_filters(task_plan, user_query):
    input_json = json.dumps(task_plan)
    try:
        resp = call_gemini(FILTER_EXTRACTOR_PROMPT, input_json, label=f'step4_{user_query[:30]}')

        result = json.loads(resp)

        # Normalize envelope: bare-array → wrapped
        if isinstance(result, list):
            print("  [filter_extractor] Normalizing bare-array response")
            result = {"tasks": result}

        if validate_filter_result(result, task_plan):
            return result
        print("  [filter_extractor] Validation failed, null predicates fallback")
        print(f"  [filter_extractor] Rejected payload: {json.dumps(result, indent=2)[:500]}")
    except Exception as e:
        print(f"  [filter_extractor] Error: {e}, null predicates fallback")
    fallback = copy.deepcopy(task_plan)
    for t in fallback['tasks']: t['predicate'] = None
    return fallback


In [60]:
# === ACTOR EXPANSION POST-PROCESSOR ===
def _expand_actor_in_predicate(pred):
    if pred is None: return None
    if 'logic' in pred:
        pred['clauses'] = [_expand_actor_in_predicate(c) for c in pred['clauses']]
        return pred
    if pred.get('field') == 'Actor':
        value = pred['value']
        op = pred['op']
        ptype = pred['type']
        return {
            "logic": "OR",
            "clauses": [
                {"field": f"Actor_{i}", "op": op, "value": value, "type": ptype}
                for i in range(1, 4)
            ]
        }
    return pred

def expand_actors_in_result(ir):
    for task in ir.get('tasks', []):
        task['predicate'] = _expand_actor_in_predicate(task.get('predicate'))
    return ir


In [61]:
# === GEOCODING RESOLVER POST-PROCESSOR ===
SF_LANDMARKS = {
    "coit tower": [37.8024, -122.4058],
    "golden gate bridge": [37.8199, -122.4783],
    "alcatraz": [37.8267, -122.4233],
    "alcatraz island": [37.8267, -122.4233],
    "union square": [37.7879, -122.4074],
    "city hall": [37.7793, -122.4193],
    "ferry building": [37.7955, -122.3937],
    "fishermans wharf": [37.8080, -122.4177],
    "fisherman's wharf": [37.8080, -122.4177],
    "pier 39": [37.8087, -122.4098],
    "lombard street": [37.8021, -122.4187],
    "transamerica pyramid": [37.7952, -122.4028],
    "palace of fine arts": [37.8028, -122.4483],
    "twin peaks": [37.7544, -122.4477],
    "dolores park": [37.7596, -122.4269],
    "alamo square": [37.7764, -122.4346],
    "chinatown": [37.7941, -122.4078],
    "north beach": [37.8060, -122.4103],
    "tenderloin": [37.7847, -122.4141],
    "nob hill": [37.7930, -122.4161],
    "russian hill": [37.8011, -122.4194],
    "pacific heights": [37.7925, -122.4352],
    "presidio": [37.7989, -122.4662],
    "the presidio": [37.7989, -122.4662],
    "golden gate park": [37.7694, -122.4862],
    "embarcadero": [37.7936, -122.3930],
    "the embarcadero": [37.7936, -122.3930],
    "treasure island": [37.8235, -122.3707],
    "fort point": [37.8106, -122.4770],
    "fort mason": [37.8060, -122.4310],
    "ghirardelli square": [37.8060, -122.4230],
    "telegraph hill": [37.8027, -122.4057],
    "ocean beach": [37.7600, -122.5100],
    "baker beach": [37.7936, -122.4836],
    "mission district": [37.7599, -122.4148],
    "the mission": [37.7599, -122.4148],
    "castro": [37.7609, -122.4350],
    "the castro": [37.7609, -122.4350],
    "soma": [37.7785, -122.3950],
    "civic center": [37.7796, -122.4157],
    "japantown": [37.7854, -122.4294],
    "oracle park": [37.7786, -122.3893],
    "moscone center": [37.7840, -122.4010],
    "grace cathedral": [37.7919, -122.4130],
    "washington square": [37.8004, -122.4104],
    "yerba buena gardens": [37.7850, -122.4025],
    "market street": [37.7853, -122.4073],
    "cable car turnaround": [37.7847, -122.4079],
    "haight ashbury": [37.7692, -122.4481],
    "haight-ashbury": [37.7692, -122.4481],
    "bernal heights": [37.7430, -122.4153],
    "potrero hill": [37.7600, -122.4002],
    "glen park": [37.7340, -122.4330],
    "cliff house": [37.7787, -122.5136],
}

def _geocode_place(name, landmarks):
    key = name.strip().lower()
    if key in landmarks: return tuple(landmarks[key])
    if key.startswith("the ") and key[4:] in landmarks: return tuple(landmarks[key[4:]])
    if ("the " + key) in landmarks: return tuple(landmarks["the " + key])
    raise ValueError(f"Could not geocode '{name}'. Not in landmarks file.")

def _resolve_geocoding_pred(pred, landmarks):
    if pred is None: return None
    if 'logic' in pred:
        for c in pred['clauses']: _resolve_geocoding_pred(c, landmarks)
        return pred
    if pred.get('type') == 'spatial' and pred.get('op') == 'within_distance':
        val = pred.get('value', {})
        if 'latitude' not in val:
            lat, lon = _geocode_place(val['reference_place'], landmarks)
            val['latitude'] = lat
            val['longitude'] = lon
    return pred

def resolve_geocoding(ir, landmarks=None):
    if landmarks is None: landmarks = SF_LANDMARKS
    for task in ir.get('tasks', []):
        _resolve_geocoding_pred(task.get('predicate'), landmarks)
    return ir


---
## Step 5: Presentation Resolver (rule-based)


In [62]:
_LOCATION_PATTERNS = [
    re.compile(r'\blocation\b', re.I), re.compile(r'\blocations\b', re.I),
    re.compile(r'\bfilming location\b', re.I), re.compile(r'\bfilming locations\b', re.I),
    re.compile(r'\bfilmed at\b', re.I), re.compile(r'\bshot at\b', re.I),
    re.compile(r'\bwhere\b', re.I), re.compile(r'\bplaces\b', re.I),
    re.compile(r'\bspots\b', re.I), re.compile(r'\bsites\b', re.I),
]
_FILM_PATTERNS = [
    re.compile(r'\bfilm\b', re.I), re.compile(r'\bfilms\b', re.I),
    re.compile(r'\btitle\b', re.I), re.compile(r'\btitles\b', re.I),
]
_METRIC_SUPPRESSORS = [
    re.compile(r'\bmost locations\b', re.I), re.compile(r'\bnumber of locations\b', re.I),
    re.compile(r'\bfewest locations\b', re.I),
    re.compile(r'\bhighest number of locations\b', re.I),
    re.compile(r'\blowest number of locations\b', re.I),
]

def _resolve_granularity(task, all_tasks):
    kind = task.get('kind', '')
    source = task.get('source', '')
    if kind in ('count', 'clarify'):
        return 'scalar'
    # Check for metric suppression
    has_metric = any(p.search(source) for p in _METRIC_SUPPRESSORS)
    has_location = any(p.search(source) for p in _LOCATION_PATTERNS)
    if has_location and not has_metric:
        return 'location'
    # Compare: inherit from deps if they agree
    if kind == 'compare':
        deps = task.get('dependsOn', [])
        dep_granularities = set()
        for t in all_tasks:
            if t['id'] in deps and 'response_granularity' in t:
                dep_granularities.add(t['response_granularity'])
        if len(dep_granularities) == 1:
            return dep_granularities.pop()
    return 'film'

def _resolve_offer_map(task):
    kind = task.get('kind', '')
    if kind in ('count', 'clarify'):
        return False
    return True  # generous policy

def resolve_presentation(ir):
    ir = copy.deepcopy(ir)
    tasks = ir.get('tasks', [])
    # Two passes: first resolve non-compare, then compare (needs deps resolved)
    for task in tasks:
        if task.get('kind') != 'compare':
            task['response_granularity'] = _resolve_granularity(task, tasks)
            task['offer_map'] = _resolve_offer_map(task)
    for task in tasks:
        if task.get('kind') == 'compare':
            task['response_granularity'] = _resolve_granularity(task, tasks)
            task['offer_map'] = _resolve_offer_map(task)
    ir['offer_map'] = any(t.get('offer_map', False) for t in tasks)
    return ir


---
## Pipeline Runner (Steps 1–5)


In [63]:
def run_preprocessing_pipeline(user_query, known_values, verbose=True):
    """Run Steps 1-5 and return the complete IR."""
    if verbose: print(f"\n{'='*60}\nQuery: {user_query}\n{'='*60}")

    # Step 1: Normalize
    norm = normalize_query(user_query, known_values)
    cleaned = norm['normalized']
    if verbose:
        print(f"\n[Step 1] Normalized: {cleaned}")
        if norm['corrections']: print(f"  Corrections: {norm['corrections']}")

    # Step 2: Safety Gate
    gate = safety_check(cleaned)
    if not gate['safe']:
        if verbose: print(f"\n[Step 2] BLOCKED: {gate['blocked_by']}")
        return {'error': True, 'message': gate['message'], 'blocked_by': gate['blocked_by']}
    if verbose: print(f"[Step 2] Safe ✓")

    # Step 3: Task Decomposer
    if verbose: print(f"\n[Step 3] Decomposing...")
    task_plan = decompose_query(cleaned)
    if verbose: print(f"  Tasks: {json.dumps(task_plan, indent=2)}")
    time.sleep(LLM_DELAY)

    # Step 4: Filter Extractor
    if verbose: print(f"\n[Step 4] Extracting filters...")
    filter_result = extract_filters(task_plan, user_query)
    filter_result = expand_actors_in_result(filter_result)
    filter_result = resolve_geocoding(filter_result)
    if verbose: print(f"  Filters: {json.dumps(filter_result, indent=2)}")
    time.sleep(LLM_DELAY)

    # Step 5: Presentation Resolver
    ir = resolve_presentation(filter_result)
    if verbose:
        print(f"\n[Step 5] Presentation resolved:")
        for t in ir['tasks']:
            print(f"  {t['id']}: granularity={t.get('response_granularity')}, offer_map={t.get('offer_map')}")
        print(f"  Top-level offer_map: {ir.get('offer_map')}")

    return ir

---
## Step 6: Code Generation (LLM)


In [64]:
# === LOAD CODE GENERATION PROMPT ===
# Upload code_generation_v2.md to Colab, then load it here.
# drive.mount('/content/drive')
CODE_GEN_PROMPT_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/code_generation_v2.2.md'

def load_codegen_prompt(path=CODE_GEN_PROMPT_PATH):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

CODEGEN_PROMPT_TEMPLATE = load_codegen_prompt()
print(f"Loaded code generation prompt: {len(CODEGEN_PROMPT_TEMPLATE)} chars, "
      f"{CODEGEN_PROMPT_TEMPLATE.count(chr(10))} lines")

# Verify injection point exists
assert '{ir_json}' in CODEGEN_PROMPT_TEMPLATE, "Prompt must contain {ir_json} placeholder"
print("✓ {ir_json} placeholder found")


Loaded code generation prompt: 55750 chars, 1579 lines
✓ {ir_json} placeholder found


In [65]:
def generate_code(ir, user_query, verbose=True):
    """
    Step 6: Generate executable Python code from the IR.
    Injects the IR into the code generation prompt and calls the LLM.
    Returns parsed {"code": ..., "explanation": ...} or error dict.
    """
    ir_json = json.dumps(ir, indent=2)
    prompt = CODEGEN_PROMPT_TEMPLATE.replace('{ir_json}', ir_json)

    if verbose:
        print(f"\n[Step 6] Generating code...")
        print(f"  IR size: {len(ir_json)} chars")
        print(f"  Prompt size: {len(prompt)} chars")

    try:
        # response_text = call_gemini(prompt, user_query)
        response_text = call_gemini(prompt, user_query, label=f'step6_{user_query[:30]}')


        # Parse the JSON response
        try:
            result = json.loads(response_text)
        except json.JSONDecodeError:
            # Try to extract JSON from response
            json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
            else:
                return {'error': True, 'message': 'Could not parse JSON from LLM response',
                        'raw': response_text[:500]}

        code = result.get('code', '')
        explanation = result.get('explanation', '')

        if not code or not code.strip():
            return {'error': True, 'message': 'LLM returned empty code field',
                    'raw': response_text[:500]}

        if verbose:
            print(f"  ✓ Code generated: {len(code)} chars")
            print(f"  Explanation: {explanation[:150]}...")

        return {'code': code, 'explanation': explanation, 'error': False}

    except Exception as e:
        return {'error': True, 'message': f'Code generation failed: {str(e)}'}


---
## Code Executor


In [66]:
import ast

def validate_code(code):
    """Basic validation before execution."""
    errors = []
    # Must parse
    try:
        ast.parse(code)
    except SyntaxError as e:
        errors.append(f"Syntax error: {e}")
        return errors

    # Must contain the function
    if 'def process_sf_film_query' not in code:
        errors.append("Missing process_sf_film_query function")

    # Must not mutate input
    if 'gdf =' in code and 'gdf_copy' not in code.split('gdf =')[0]:
        errors.append("Possible mutation of input gdf")

    return errors

def execute_code(code, gdf, verbose=True):
    """Execute generated code against the GeoDataFrame."""
    # Validate first
    validation_errors = validate_code(code)
    if validation_errors:
        return {
            'success': False,
            'data': None,
            'summary': f"Validation failed: {'; '.join(validation_errors)}",
            'metadata': {'errors': validation_errors}
        }

    # Execute
    try:
        exec_globals = {}
        exec(code, exec_globals)

        if 'process_sf_film_query' not in exec_globals:
            return {
                'success': False,
                'data': None,
                'summary': 'Function process_sf_film_query not found after execution',
                'metadata': {}
            }

        func = exec_globals['process_sf_film_query']
        result = func(gdf)

        # NEW: detect errors the generated code caught internally.
        # The code generator's try/except returns a structured error payload
        # (data=None, metadata.error set) instead of raising. Treat that as
        # a failure, not a success.
        internal_error = (
            isinstance(result, dict)
            and 'error' in result.get('metadata', {})
        )

        if internal_error:
            if verbose:
                err = result['metadata'].get('error', 'unknown')
                err_type = result['metadata'].get('error_type', 'Error')
                print(f"\n[Execution] ✗ Internal error: {err_type}: {err}")
            return {
                'success': False,
                'data': result.get('data'),
                'summary': result.get('summary', 'Internal execution error'),
                'metadata': result.get('metadata', {})
            }

        if verbose:
            print(f"\n[Execution] ✓ Success")
            if isinstance(result, dict):
                print(f"  Summary: {result.get('summary', 'N/A')}")
                data = result.get('data')
                if isinstance(data, (pd.DataFrame, gpd.GeoDataFrame)):
                    print(f"  Data: DataFrame with {len(data)} rows")
                elif isinstance(data, dict):
                    print(f"  Data keys: {list(data.keys())[:10]}")
                elif data is not None:
                    print(f"  Data: {str(data)[:200]}")
            else:
                print(f"  Result: {str(result)[:200]}")

        return {'success': True, 'result': result}

    except Exception as e:
        if verbose:
            print(f"\n[Execution] ✗ Error: {type(e).__name__}: {e}")
        return {
            'success': False,
            'data': None,
            'summary': f"Execution error: {str(e)}",
            'metadata': {'error': str(e), 'error_type': type(e).__name__}
        }


---
## End-to-End Runner


In [67]:
def run_full_pipeline(user_query, gdf, known_values, verbose=True):
    """
    Complete pipeline: preprocessing (Steps 1-5) → code generation → execution.
    Returns all intermediate results for inspection.
    """
    pipeline_result = {
        'query': user_query,
        'ir': None,
        'codegen': None,
        'execution': None,
        'success': False,
    }

    # Steps 1-5: Preprocessing
    ir = run_preprocessing_pipeline(user_query, known_values, verbose=verbose)

    # Check for blocked queries
    if isinstance(ir, dict) and ir.get('error'):
        pipeline_result['ir'] = ir
        if verbose: print(f"\n❌ Pipeline stopped: {ir.get('message')}")
        return pipeline_result

    # Check for clarify-only tasks
    if all(t['kind'] == 'clarify' for t in ir.get('tasks', [])):
        pipeline_result['ir'] = ir
        if verbose: print(f"\n⚠️ Clarification needed — skipping code generation")
        return pipeline_result

    pipeline_result['ir'] = ir
    time.sleep(LLM_DELAY)

    # Step 6: Code Generation
    codegen = generate_code(ir, user_query, verbose=verbose)
    pipeline_result['codegen'] = codegen

    if codegen.get('error'):
        if verbose: print(f"\n❌ Code generation failed: {codegen.get('message')}")
        return pipeline_result

    # Execution
    execution = execute_code(codegen['code'], gdf, verbose=verbose)
    pipeline_result['execution'] = execution
    pipeline_result['success'] = execution.get('success', False)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Pipeline {'✓ SUCCESS' if pipeline_result['success'] else '✗ FAILED'}")
        print(f"{'='*60}")

    return pipeline_result


---
## Test Suite — Phase 1 (Single-Task Queries)


In [68]:
PHASE_1_TESTS = [
    {
        "name": "T1: Retrieve by Director (string ==)",
        "query": "films directed by alfred hitchcock",
        "expect": "retrieve, film granularity, Director predicate",
    },
    {
        "name": "T2: Retrieve by Location (contains)",
        "query": "films shot on market street",
        "expect": "retrieve, film granularity, Locations contains",
    },
    {
        "name": "T3: Retrieve by Actor (virtual field)",
        "query": "films with sean penn",
        "expect": "retrieve, film granularity, Actor OR expansion",
    },
    {
        "name": "T4: Count films in a decade",
        "query": "how many films were shot in the 80s",
        "expect": "count, scalar granularity, Year between 1980-1989",
    },
    {
        "name": "T5: Rank directors by film count",
        "query": "top 5 directors with the most films",
        "expect": "rank, film granularity, no predicate filter (null or all rows)",
    },
    {
        "name": "T6: Retrieve by Year (exact)",
        "query": "films from 1985",
        "expect": "retrieve, film granularity, Year == 1985",
    },
    {
        "name": "T7: Retrieve by Director + Year range (AND)",
        "query": "films by alfred hitchcock from the 1950s",
        "expect": "retrieve, film granularity, Director AND Year between",
    },
    {
        "name": "T8: Spatial retrieve (within_distance)",
        "query": "films within 1 mile of coit tower",
        "expect": "retrieve, location granularity, geometry within_distance",
    },
    {
        "name": "T9: Retrieve locations for a film",
        "query": "show filming locations for vertigo",
        "expect": "retrieve, location granularity, Title contains vertigo",
    },
    {
        "name": "T10: Rank actors by film count in decade",
        "query": "actors who appeared in the most films in the 90s",
        "expect": "rank, film granularity, Year between 1990-1999",
    },
]


In [69]:
def run_test_suite(tests, gdf, known_values, pause=2):
    """Run a suite of test queries through the full pipeline."""
    results = []
    total = len(tests)

    for i, test in enumerate(tests):
        print(f"\n{'#'*60}")
        print(f"  TEST {i+1}/{total}: {test['name']}")
        print(f"  Query: {test['query']}")
        print(f"  Expected: {test['expect']}")
        print(f"{'#'*60}")

        result = run_full_pipeline(test['query'], gdf, known_values, verbose=True)
        result['test_name'] = test['name']
        result['expected'] = test['expect']
        results.append(result)

        if i < total - 1:
            print(f"\n  ⏳ Waiting {pause}s before next test...")
            time.sleep(pause)

    # Summary
    print(f"\n\n{'='*60}")
    print(f"TEST SUITE SUMMARY")
    print(f"{'='*60}")
    passed = sum(1 for r in results if r['success'])
    failed = total - passed
    print(f"  Passed: {passed}/{total}")
    print(f"  Failed: {failed}/{total}")
    print()
    for r in results:
        status = "✓" if r['success'] else "✗"
        print(f"  {status} {r['test_name']}")
        if not r['success']:
            if r.get('codegen', {}).get('error'):
                print(f"    → Codegen error: {r['codegen'].get('message','')[:100]}")
            elif r.get('execution'):
                print(f"    → Execution error: {r['execution'].get('summary','')[:100]}")

    return results


---
## Run Tests

Uncomment and run the cells below to execute the test suite.


In [ ]:
# --- Run a single query first to smoke test ---
result = run_full_pipeline("top 5 directors with the most films", gdf, known_values)



Query: top 5 directors with the most films

[Step 1] Normalized: top 5 directors with the most films
[Step 2] Safe ✓

[Step 3] Decomposing...
  Tasks: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "top 5 directors with the most films",
      "dependsOn": []
    }
  ]
}

[Step 4] Extracting filters...
Successfully saved to /content/drive/MyDrive/Colab Notebooks/SF Film Project/calude_refactoring/e2e-testing/raw_responses/20260420_015218_step4_top_5_directors_with_the_most_.json
  Filters: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "top 5 directors with the most films",
      "dependsOn": [],
      "predicate": null
    }
  ]
}

[Step 5] Presentation resolved:
  t1: granularity=film, offer_map=True
  Top-level offer_map: True

[Step 6] Generating code...
  IR size: 262 chars
  Prompt size: 56003 chars


KeyboardInterrupt: 

In [ ]:
# --- Inspect the generated code ---
if result.get('codegen') and not result['codegen'].get('error'):
    print(result['codegen']['code'])


def process_sf_film_query(gdf):
    """
    Execute a read-only query against the SF film locations GeoDataFrame.
    Returns a standardized result dictionary.
    """
    import pandas as pd
    import numpy as np
    import geopandas as gpd

    def clean_column_data(series):
        """Remove empty/null/whitespace/stringy-null values from a string Series.
        Use ONLY for final output cleanup, NEVER for building boolean masks.
        """
        stringified = series.astype(str)
        exclude = (
            series.isna() |
            (stringified.str.strip() == '') |
            (stringified.str.lower().isin(['none', 'nan', 'null']))
        )
        return series[~exclude]

    try:
        gdf_copy = gdf.copy()
        task_results = {}

        # ── Task t1: retrieve top 5 directors with the most films ──
        # Predicate is null, so matched_rows is the full dataset.
        t1_matched_rows = gdf_copy

        # Deduplicate to film level before counting directors
    

In [70]:
# --- Run the full Phase 1 test suite ---
test_results = run_test_suite(PHASE_1_TESTS, gdf, known_values)



############################################################
  TEST 1/10: T1: Retrieve by Director (string ==)
  Query: films directed by alfred hitchcock
  Expected: retrieve, film granularity, Director predicate
############################################################

Query: films directed by alfred hitchcock

[Step 1] Normalized: films directed by alfred hitchcock
[Step 2] Safe ✓

[Step 3] Decomposing...
  Tasks: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films directed by alfred hitchcock",
      "dependsOn": []
    }
  ]
}

[Step 4] Extracting filters...
Successfully saved to /content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/raw_responses/20260420_195251_step4_films_directed_by_alfred_hitch.json
  [filter_extractor] Normalizing bare-array response
  Filters: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films directed by alfred hitchcock",
      "dependsOn": [],
   

---
## Inspect Individual Results

After running tests, inspect specific results:


In [ ]:
# --- View IR for a specific test ---
idx = 1  # change to inspect different test
print(json.dumps(test_results[idx]['ir'], indent=2))


{
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films shot on market st",
      "dependsOn": [],
      "predicate": {
        "field": "Locations",
        "op": "contains",
        "value": "market st",
        "type": "attribute"
      },
      "response_granularity": "film",
      "offer_map": true
    }
  ],
  "offer_map": true
}


In [25]:
import json

output_data = []

for i in range(10): # For T1 to T10
    test_name_prefix = f"T{i+1}"
    # Find the corresponding test result from PHASE_1_TESTS using test_name
    # The test_results list is ordered the same as PHASE_1_TESTS
    test_idx = i

    if test_idx < len(test_results):
        result = test_results[test_idx]
        generated_code = "NO_GEN_CODE_AVAILABLE"
        if result.get('codegen') and not result['codegen'].get('error'):
            generated_code = result['codegen']['code']

        output_data.append({str(test_name_prefix): generated_code})
    else:
        output_data.append({str(test_name_prefix): "NO_GEN_CODE_AVAILABLE"})

output_filename = "_gen_code_for_T1_to_T10_tests.json"
with open(output_filename, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"Generated code for tests T1-T10 written to {output_filename}")


Generated code for tests T1-T10 written to _gen_code_for_T1_to_T10_tests.json


In [ ]:
# --- View generated code for a specific test ---

idx = 1
if test_results[idx].get('codegen'):
    print(test_results[idx]['codegen']['code'])


KeyError: 'code'

In [ ]:
# --- View execution result for a specific test ---
idx = 1
if test_results[idx].get('execution'):
    print(json.dumps(test_results[idx]['execution'].get('result', {}),
          indent=2, default=str)[:2000])


{
  "data": [
    {
      "id": 49,
      "Title": "The Net",
      "Year": 1995,
      "Locations": "Montgomery & Market St",
      "Fun_Facts": "",
      "Director": "Irwin Winkler",
      "Writer": "John Brancato",
      "Actor_1": "Sandra Bullock",
      "Actor_2": "Jeremy Northam",
      "Actor_3": "",
      "geometry": "POINT (-122.40166 37.78921)"
    },
    {
      "id": 55,
      "Title": "Terminator - Genisys",
      "Year": 2015,
      "Locations": "1 Market St Landmark Building",
      "Fun_Facts": "",
      "Director": "Alan Taylor",
      "Writer": "James Cameron",
      "Actor_1": "Arnold Schwarzenegger",
      "Actor_2": "Jason Clarke",
      "Actor_3": "",
      "geometry": "POINT (-122.39499 37.79398)"
    },
    {
      "id": 68,
      "Title": "The Game",
      "Year": 1997,
      "Locations": "Sheraton Palace Hotel (639 Market St)",
      "Fun_Facts": "The hotel was destroyed in the 1906 earthquake and fire, had to be rebuilt, and was reopened in 1909.",
      "Dir

# analysis of the ten tests

## Overall  April-17 first test results

Labels now match the log cleanly. I went through all ten. Here's what I found, test by test.

**T1 (Hitchcock director) — the failure is a repetition loop.** The generated body is literally just the opening comment `# ── Task t1 ──` followed by ~600 consecutive `─` characters and then the file ends. No code was emitted after the banner. The `try:` on line 26 has no body, which is why the validator reported "expected except or finally block." The LLM got stuck in a token-repetition loop on the box-drawing character and hit max_output_tokens before writing any logic. I strongly suspect the null predicate (fallback from the 503) is the trigger — with the predicate gone, the prompt has fewer anchors, and Gemini drifted into drawing the divider forever. Removing the decorative `─` characters from your skeleton example in the prompt (or replacing with plain `#` dashes) would likely prevent this.

**T2 (market street) — the f-string breaks on a real newline.** Line 62 reads `f" For example: {', '.join(sample_titles[:-1])}, and {sample_titles[-1]}.` and then an actual newline character sits inside the string before the closing quote on line 63. Python single-quoted f-strings can't contain literal newlines. The mechanism is the JSON round-trip: the model emitted `\n` as an escape inside its JSON response, `json.loads` correctly decoded that to a real newline, and now the code text has a raw newline inside a string literal. The predicate logic itself is fine — `str.contains('market street', case=False)` is the right translation of `op: contains`. Fix is in the prompt: forbid embedded newlines in string literals, or instruct the model to use explicit `\\n` / triple-quoted strings for multi-line output.

**T3 (Sean Penn) — passes but silently swaps the operator.** IR says `op: "=="` with value `"sean penn"`. Generated code uses `str.contains('sean penn', case=False)` across the three actor columns. The OR expansion is correct, the row result is correct (5 films), but the predicate was translated as substring-match, not equality. Works here because no "Sean Penny" exists in the data. This is a category-level bug I'll come back to below.

**T4 (count in the 80s) — clean.** `pd.to_numeric(gdf_copy['Year'], errors='coerce').between(1980, 1989, inclusive='both')` is the right translation. Dedupes to film level before counting, which matches "how many films." The 39 result looks plausible; hand-verify to be sure.

**T5 (top 5 directors) — passes, but the IR is wrong and the codegen is covering for it.** The Task Decomposer labeled this `kind: "retrieve"`, not `kind: "rank"`. The codegen LLM read the `source` text "top 5 directors with the most films," ignored the IR kind, and wrote a rank operation anyway. The output is correct but the IR contract is violated — you have a Step 3 bug hiding behind a Step 6 win. Also, line 31 `t1_matched_rows = gdf_copy` is an aliasing, not a copy — no harm here because nothing mutates it, but the hard invariants mention immutability.

**T6 (films from 1985) — clean.** `== 1985` on coerced numeric is the right translation of `op: "=="` on Year.

**T7 (Hitchcock + 1950s AND) — passes with the same operator swap as T3.** Director clause is `str.contains('alfred hitchcock', case=False)` instead of equality. Year clause is correct `.between(1950, 1959)`. AND combination is structurally right. One stylistic note: T7 layers `.notna() & .strip().ne('') & ~.isin(['none','nan','null']) & .contains(...)` while T3 uses a compact `.astype(str).apply(...)`. The prompt hasn't picked a canonical null-handling idiom, so each generation improvises.

**T8 (coit tower, 1 mile) — the "silent success" we discussed.** Line 107: `task_results['t1']['predicate']['value']['reference_place']`. The `task_results['t1']` dict only has keys `matched_rows`, `text_view`, `result` — there is no `predicate` key, because the hard invariant forbids stashing the IR in runtime state. That access raises `KeyError: 'predicate'`, the except block catches it, and the pipeline marks the run successful because the function returned without raising. The whole spatial body above that line is actually well-written — reprojects to EPSG:32610 (UTM 10N, correct for SF), converts 1 mile to meters, applies the mask back against the original-CRS frame. If the model had written `"within 1 mile of coit tower"` as a literal string (which it already knew at prompt time), this would have passed for real. The fix is a sentence in the prompt: *values from the predicate that you want to echo in the summary must be inlined as literals, not read from task_results at runtime.*

**T9 (Vertigo locations) — passes, same operator swap.** IR: `op: "=="`, `value: "vertigo"`. Code: `str.contains('vertigo', case=False)`. Returns 16 rows which is correct for the dataset, but would also pick up a hypothetical "Vertigo 2." Granularity is right: no dedup because `response_granularity` is `"location"`.

**T10 — no code generated.** 429 rate limit, not a codegen issue.

---

**The three patterns worth fixing at the prompt/pipeline level, ranked by severity:**

The first pattern is the **silent `==` → `contains` swap** on string fields (T3, T7, T9 — three of the seven passing tests). This is systematic. The LLM is compensating for Step 4 lowercasing predicate values — it correctly infers that the column has "Sean Penn" while the IR says "sean penn," so it reaches for case-insensitive substring. The fix is upstream: either Step 4 preserves original case, or the codegen prompt adds a rule that `==` on string fields means `series.str.lower() == value.lower()` — exact, not substring. This is the bug most likely to cause a subtle wrong answer on a real query ("sean penn" matching "Sean Penny").

The second is **runtime IR access** (T8), which is already forbidden by the hard invariants but the model did it anyway. Needs a concrete counter-example in the prompt showing "to build a summary string, inline the literal value directly, like `f\"within 1 mile of coit tower\"` — do not read it from task_results."

The third is the **success-detection gap in `execute_code`**. Even if codegen is fixed, errors caught by the generated try/except still return a structured payload; the pipeline should treat `result['metadata'].get('error')` or `result['data'] is None` as failure signals, not just raised exceptions. T8 was your canary for this.

T1's repetition loop and T2's f-string newline are both real bugs but narrower — remove the `─` from the skeleton, forbid literal newlines inside string literals, and both should disappear.

Want me to draft the prompt deltas for the three systemic issues?



## T1

**manual testing query:**
```
result = gdf[
    gdf['Director'].astype(str).str.lower().str.contains('hitchcock', na=False)
].drop_duplicates(subset=['Title', 'Year'])

```

**me**: Reharding T1 --  this is from the logs that I shared earlier above in my first message on this thread ... ::

[Step 4] Extracting filters...   [filter_extractor] Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently

... so it makes sense that its generated code is faulty, no?


**claude:** You're right that the 503 is the root cause of the chain — but I'd push back gently on "so the faulty code makes sense."

The 503 gave codegen a `predicate: null` IR. That's a **valid** IR state, not a malformed one. Look at T5: same `kind: retrieve`, same `predicate: null`, same empty `dependsOn` — and T5 produced clean, working code. So null-predicate-retrieve isn't inherently broken at the codegen step. Something else about T1 specifically sent the model into that dash-drawing loop.

My guess at what's different: T5's source ("top 5 directors with the most films") is self-consistent with `predicate: null` — a ranking doesn't need a filter. T1's source ("films directed by alfred hitchcock") screams "there should be a filter here" but the IR says there isn't one. The model got a contradictory instruction — *retrieve films by Hitchcock, but don't filter anything* — and instead of picking a lane (either "apply my own filter from the source text" or "return all rows and note the mismatch"), it stalled and started drawing the divider.

So two things are true at once:
1. A healthy filter_extractor would have produced a populated predicate, and this specific failure would not have happened. Rerunning T1 on a good day likely passes.
2. Codegen still has a latent failure mode: when the IR and the `source` text disagree, it doesn't handle the contradiction gracefully. That's worth knowing about even if you never see it again in this test suite, because real users will occasionally produce queries where Step 4 extracts something wrong or incomplete.

If you want to de-prioritize T1 as "infra noise, not a real codegen bug" — reasonable call, especially since T5 is your control case showing the null-predicate path works. Just worth flagging that the contradiction-handling behavior is a separate axis from the 503.

**me**:Ideally, since the failure happend at IR, I need to return or ask the user to re-submit in a few seconds or such. Such a safeguard againast the API's 503 or 429 must be added eventually!

**claude** :

Yes, exactly — and this is really a resilience layer around your LLM calls, not a codegen concern. Once you add it, T1-style cascades go away entirely.

A few design points worth thinking through:

**Retry inside the step vs. surface to the user.** A 503 ("high demand, temporary") is a good candidate for automatic retry with backoff — the user never needs to know. A 429 with `retryDelay: 21s` is also technically retryable, but now you're asking the user to wait 21 seconds staring at a spinner, which is worse UX than a clean "service is busy, try again shortly." I'd split behavior by error code: 503 retries silently (2–3 attempts, exponential backoff), 429 surfaces immediately with the suggested wait time from the response payload.

**The silent fallback is the bigger problem than the API error.** Right now Step 4 catches the 503, logs it, and returns `predicate: null` — which *looks* like a legitimate "no filter needed" signal downstream. Step 5 and Step 6 have no way to distinguish "the extractor genuinely decided there's no predicate" (T5) from "the extractor crashed and punted" (T1). The fix isn't just retry — it's to stop pretending the call succeeded. When the extractor exhausts retries, return a sentinel the pipeline recognizes as failure: `{"error": "extractor_unavailable", "retry_after": 21, "step": 4}`, and let `run_full_pipeline` short-circuit the same way it does for safety-gate blocks or clarify-only tasks.

**Make it uniform across the three LLM steps.** Step 3, Step 4, and Step 6 all call Gemini. Right now I can see Step 4 has this bespoke null-fallback, Step 6 just re-raises (T10 in your log). You probably want a single wrapper — `call_gemini_with_retry(prompt, max_retries=3, step_name=...)` — that every LLM step uses. One place to tune backoff, one place to handle 429 vs 503 vs other errors, one place to format the user-facing message.

**What the user sees.** A friendly message with a concrete retry suggestion is better than a generic "something went wrong." If the payload gives you `retryDelay: 21s`, use it: *"The service is briefly rate-limited — please try again in about 25 seconds."* Rounding up slightly prevents a second immediate 429.

**One gotcha on the free-tier 429.** Your T10 hit the daily quota (`generate_content_free_tier_requests, limit: 20`), not a minute-level throttle. Retrying won't help — that's out until the quota resets. Worth distinguishing per-minute throttles (retry-friendly) from daily-quota exhaustion (message says "try again tomorrow or upgrade"). The response details include `quotaId` which tells you which one.

Do you want to keep this as a single mid-layer wrapper, or lift it up so it's the pipeline orchestrator's job to handle step-level failures? I'd lean wrapper — closer to the fault, easier to test in isolation — but the orchestrator still needs to know how to react to a structured failure bubbling up.